# 논문 A 「종량 하한과 가격 과소신고: 한국 농산물 선택 세율의 증거」 — 분석 파이썬 코드

이 노트북은 논문 A(`종량하한과_과소신고.md`)의 본문·표 수치를 만드는 파이썬 코드다. 셀은 세 논문 공용 셀 창고 `cache/cells.py`에서 논문 A의 몫만(§1·§1b 일부·§3·§4b·4c·4e 일부·4g·§5 일부·§6 EXPECT_A) 빌더 `cache/build_nb_paperA.py`가 골라 만든 것이며, 첫째 질문의 격차 회귀(§2)와 혼합 역산·감시형 세분·적응 속도(§4a·4d·4f·4h·4i)는 논문 B의 것이라 없다. `research/`에서 위에서 아래로 실행하면(약 5분) `outputs/`의 표와 무역통계 DB만 읽고, 마지막 셀이 논문이 인용한 수치 389건을 `EXPECT_A`(라벨 → (인용값, 허용 오차))로 대조해 하나라도 어긋나면 멈춘다. 등록값은 `outputs/논문A_수치.json`, 결과는 `outputs/논문A_검증_결과.json`에 남는다.

입력 표를 만든 스크립트는 저장소에 파일로 있다. 세율 두 화면 `scripts/01_fetch_tariff.py`, 실행세율과 종량 하한 `scripts/02_build_applied_rate.py`, 양허관세 별표 1 `research/scripts/25_parse_concession_annex.py`, 환율 `26_fetch_ecos_fx.py`, 코드 쌍 `16_build_code_pairs.py`, 품목 상세 `17_fetch_item_detail.py`, 신설 코드 처리군 `18_build_treatments.py`, 원산지별 실행세율 `19_extend_applied_rate.py`, 통관 패널 `20_build_panel.py`, 미러 통계 `21_fetch_comtrade_mirror.py`·`22_mirror_gap.py`. 사전세액심사 목록(브라우저 자바스크립트)·유통이력 별표(HWP·HWPX 읽기)·파일럿 계열(`pilot_series.csv`)은 대화형으로 만들었고 그 스니펫은 `자료구축/프로그램.ipynb`와 `cache/cells.py`(`PILOT_CELL`)에 있다.

In [1]:
import os, re, json, warnings, glob
import numpy as np, pandas as pd
import duckdb
import statsmodels.api as sm
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)
# 이 노트북은 KCSTARIFF/research/ 에서 연다. 무역통계(KCSDB2)는 환경변수 KCSDB2_ROOT 또는 기본 경로에서 읽는다.
A = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(A, "outputs", "panel")):
    up = os.path.dirname(A)
    if up == A: raise FileNotFoundError("research/outputs/panel 을 찾지 못했습니다")
    A = up
O = os.path.join(A, "outputs"); P = os.path.join(O, "panel")
KCSDB2 = os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2")
KCS = os.path.join(KCSDB2, "data", "processed", "kcsdb.duckdb"); TAR = os.path.join(os.path.dirname(A), "data", "processed", "kcstariff.duckdb")
CHK = {}
def chk(label, value, digits=2):
    """본문 인용 수치를 등록한다. §6에서 문서와 대조."""
    CHK[label] = round(float(value), digits) if isinstance(value, (int, float, np.floating, np.integer)) else value
    return CHK[label]

def fe_ols(df, y, xcols, fe, cluster, tol=1e-9, maxit=200):
    """다중 고정효과 OLS: 교대 사영으로 중심화한 뒤 OLS, 군집 표준오차(cluster)."""
    d = df[[y] + xcols + fe + [cluster]].dropna().copy()
    Z = d[[y] + xcols].to_numpy(dtype=float)
    groups = [d[f].astype("category").cat.codes.to_numpy() for f in fe]
    for _ in range(maxit):
        Z0 = Z.copy()
        for g in groups:
            m = np.zeros((g.max() + 1, Z.shape[1])); n = np.bincount(g)
            np.add.at(m, g, Z); Z = Z - m[g] / n[g][:, None]
        if np.abs(Z - Z0).max() < tol: break
    yv, X = Z[:, 0], Z[:, 1:]
    XtX_inv = np.linalg.pinv(X.T @ X); b = XtX_inv @ X.T @ yv; e = yv - X @ b
    cl = d[cluster].astype("category").cat.codes.to_numpy(); G = cl.max() + 1
    S = np.zeros((G, X.shape[1])); np.add.at(S, cl, X * e[:, None]); meat = S.T @ S
    n, k = X.shape; V = XtX_inv @ meat @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    se = np.sqrt(np.diag(V)); t = b / se
    return pd.DataFrame({"coef": b, "se": se, "t": t, "within_sd": X.std(axis=0)}, index=xcols).assign(n=n, clusters=G)
print("research:", A); print("KCSDB2:", KCSDB2)

research: C:\Work\Projects\KCSTARIFF\research
KCSDB2: C:\Work\Projects\KCSDB2


### §1 자료 — 패널·코드 쌍·처리군·집행 지정 목록·환율

In [2]:
pc = pd.read_parquet(os.path.join(P, "panel_codes.parquet"))
pp = pd.read_parquet(os.path.join(P, "panel_pairs.parquet"))
pairs = pd.read_csv(os.path.join(O, "코드쌍_목록.csv"), dtype={"hs10_high": str, "hs10_low": str})
pairs = pairs[(pairs.확인.fillna("") == "Y") | pairs.source.isin(["별표", "별표(선택)"])].copy()
pairs["src"] = np.where(pairs.source.isin(["별표", "별표(선택)"]), "별표", "규칙")
pairs["agri"] = pairs.hs10_high.str[:2].astype(int) <= 24
treat = pd.read_csv(os.path.join(O, "처리군_목록.csv"), dtype={"hs10": str, "mate_hs10": str, "pred_codes": str})
pre = pd.read_csv(os.path.join(O, "사전세액심사_대상_이력_2016_2026.csv"), dtype=str)
dist = pd.read_csv(os.path.join(O, "유통이력_신고물품_통합_구간_2009_2026.csv"), dtype=str)
fx = pd.read_csv(os.path.join(O, "환율_월별_USDKRW.csv"))
fxy = fx.groupby("year").krw_per_usd.mean()
print(f"panel_codes {len(pc):,} / panel_pairs {len(pp):,} / 쌍 {len(pairs)} (별표 {int((pairs.src=='별표').sum())}, 규칙 {int((pairs.src=='규칙').sum())}) / 처리군 {len(treat)}")
chk("쌍 수", len(pairs), 0); chk("쌍 코드 수", len(set(pairs.hs10_high) | set(pairs.hs10_low)), 0)
chk("쌍 별표", int((pairs.src == "별표").sum()), 0); chk("쌍 규칙", int((pairs.src == "규칙").sum()), 0)
chk("패널 코드 수", pc.hs10.nunique(), 0); chk("패널 원산지 수", pc.stat_cd.nunique(), 0)
for (rv, ty), n in treat.groupby(["rev", "type"]).size().items(): chk(f"처리군 {rv} {ty}", n, 0)
for st, n in treat[treat.type.str.startswith("감시형")].groupby("sub_type").size().items(): chk(f"감시형 sub_type {st}", n, 0)
chk("사전세액심사 코드 수", pre.hs10.nunique(), 0); chk("유통이력 코드 수", dist.hs10.nunique(), 0); chk("유통이력 구간 수", len(dist), 0)
cg = pd.read_csv(os.path.join(O, "패널_연속성_그룹.csv"))
cg1 = cg[cg.before_musd >= 1]
chk("연속성 집단 수", len(cg), 0); chk("연속성 집단(1백만 달러 이상)", len(cg1), 0); chk("연속성 비율 중위", cg1.ratio.median(), 2); chk("연속성 비율 10분위", cg1.ratio.quantile(.1), 2); chk("연속성 비율 90분위", cg1.ratio.quantile(.9), 2)
chk("연속성 총액 비율", cg1.after_musd.sum() / cg1.before_musd.sum(), 2); chk("잔여 통관 최대 개월", cg.residual_months.max(), 0)

panel_codes 639,188 / panel_pairs 1,070,404 / 쌍 444 (별표 358, 규칙 86) / 처리군 1462


8.0

In [3]:
# 1b 자료 수치(셀 창고 §1b 가운데 논문 A가 인용하는 줄만): 세율 DB·양허 별표·품목 상세·미러·수집 검증 CSV
con1 = duckdb.connect(); con1.execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)"); con1.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)")
ar = con1.sql("SELECT * FROM tr.fct_applied_rate").df()
chk("자료 하한 있는 행", int(ar.floor_won_kg.notna().sum()), 0)
by = pd.read_csv(os.path.join(O, "양허관세_별표1_2025.csv"), dtype={"hs10": str})
chk("자료 별표 1의 가 행", int((by.byeolpyo == "1의 가").sum()), 0); chk("자료 별표 1의 나 행", int((by.byeolpyo == "1의 나").sum()), 0)
ho = by[by.higher_of.astype(str).str.lower() == "true"]
chk("자료 종량 대안 코드", ho.hs10.nunique(), 0); chk("자료 종량 대안 코드 가", ho[ho.byeolpyo == "1의 가"].hs10.nunique(), 0); chk("자료 종량 대안 코드 나", ho[ho.byeolpyo == "1의 나"].hs10.nunique(), 0)
fl_ = ho.specific_won_kg / (ho.adval / 100); chk("자료 하한 최소 원", fl_.min(), 0); chk("자료 하한 최대 원", fl_.max(), 0)
dd = pd.read_csv(os.path.join(O, "품목상세_세율_2012_2026.csv"), dtype={"hs10": str, "year": int})
seven = ["FCN1", "FEU1", "FUS1", "FAS1", "FIN1", "FVN1", "FCA1"]
mr = con1.sql("SELECT year, hs10, rate_cd, adval FROM tr.tariff_rate WHERE source='main' AND rate_cd IN ('FCN1','FEU1','FUS1','FAS1','FIN1','FVN1','FCA1') AND month(valid_from)=1 AND day(valid_from)=1").df()
cmp_ = dd[dd.rate_cd.isin(seven)].merge(mr, on=["year", "hs10", "rate_cd"])
chk("자료 일곱 상대 대조 행", len(cmp_), 0); chk("자료 일곱 상대 대조 어긋남", int((cmp_.adval_first.astype(float).round(2) != cmp_.adval.round(2)).sum()), 0)
chk("자료 품목 상세 코드", dd.hs10.nunique(), 0); chk("자료 품목 상세 구분기호", dd.rate_cd.nunique(), 0); chk("자료 선택1 협정", dd[dd.rate_cd.str.match(r"^F[A-Z]+1$")].rate_cd.nunique(), 0)
cm_ = pd.read_csv(os.path.join(O, "comtrade_mirror_hs6_2012_2024.csv"), dtype={"cmdCode": str}); chk("자료 미러 행", len(cm_), 0); chk("자료 미러 보고국", cm_.reporter.nunique(), 0); chk("자료 미러 HS6", cm_.cmdCode.nunique(), 0)
REPO = os.path.dirname(A)
vc = pd.read_csv(os.path.join(REPO, "outputs", "수집_검증.csv"), dtype={"key": str})
for cd in ["A", "C"]:
    chk(f"자료 두 화면 일치율 {cd}", float(vc[vc.item == f"두 화면 일치율 {cd}"].value.iloc[0]), 2); chk(f"자료 두 화면 대조 쌍 {cd}", float(vc[vc.item == f"두 화면 대조 쌍 {cd}"].value.iloc[0]), 0)
vs = pd.read_csv(os.path.join(O, "실행세율_원산지별_검증_요약.csv")).set_index("item").value; chk("자료 19 일곱 상대 대조 행", float(vs["일곱 상대 대조 행"]), 0); chk("자료 19 일곱 상대 어긋남", float(vs["일곱 상대 어긋남"]), 0)

0.0

### §3 종량 하한 — 하한 아래 몫(표 2)과 건고추 저가 신고 사건, 변환비 쌍의 단가 비율, 미러(표 3·표 7과 후보 대조), 파일럿 계열(건조생강·건고추)

In [4]:
fl = pc[pc.floor_won_kg.notna() & (pc.imp_wgt > 0)].merge(fx[["yyyymm", "krw_per_usd"]], on="yyyymm", how="left")
fl["uv_won"] = fl.uv * fl.krw_per_usd; fl["below"] = fl.uv_won < fl.floor_won_kg
fl["ratio"] = fl.uv_won / fl.floor_won_kg
def wshare(g): return np.average(g.below, weights=g.imp_dlr)
t8 = fl[fl.year >= 2012].groupby(["hs10"]).apply(lambda g: pd.Series(dict(코드수=1, 수입_백만달러=g.imp_dlr.sum()/1e6, 하한_원kg=g.floor_won_kg.iloc[-1], 하한아래_금액몫=wshare(g), 중국_하한아래=wshare(g[g.stat_cd=="CN"]) if (g.stat_cd=="CN").any() else np.nan, 비율중위=g.ratio.median()))).reset_index()
names = pd.read_csv(os.path.join(KCSDB2, "data", "external", "HSK_별표", "HSK_별표_2025.csv"), dtype=str).set_index("code").leaf
t8["품명"] = t8.hs10.map(names)
t8 = t8.sort_values("수입_백만달러", ascending=False)
display(t8.head(25).round(3))
print("하한 코드 수", len(t8), " 금액 가중 하한 아래 몫:", round(np.average(t8.하한아래_금액몫, weights=t8.수입_백만달러), 3))
chk("하한 코드 수", len(t8), 0); chk("하한 아래 몫 전체", np.average(t8.하한아래_금액몫, weights=t8.수입_백만달러), 3)
for h in ["0904210000", "0910112000", "1207400000", "1201909000"]:
    if h in set(t8.hs10): chk(f"하한 아래 몫 {h}", float(t8.set_index("hs10").loc[h, "하한아래_금액몫"]), 3)
t8i = t8.set_index("hs10")
for h in ["1201901000", "1201909000", "1207400000", "1207991000", "0904210000", "0409000000", "0703209000"]:
    if h in t8i.index:
        chk(f"하한 표 {h} 수입", t8i.loc[h, "수입_백만달러"], 0); chk(f"하한 표 {h} 하한", t8i.loc[h, "하한_원kg"], 0); chk(f"하한 표 {h} 몫", t8i.loc[h, "하한아래_금액몫"], 3); chk(f"하한 표 {h} 비율중위", t8i.loc[h, "비율중위"], 2)
# 건고추 저가 신고 사건(조세심판원 2024-05-21 결정, 조심 2023관0105; 논문 A IV.1): 결정문의 신고 단가 1,600달러/톤을 2022-11 월평균 환율로 원/kg으로 바꿔 종량 하한·종량세와 비교한다
fx2211 = float(fx[fx.yyyymm == 202211].krw_per_usd.iloc[0]); chk("환율 2022.11", fx2211, 1)
uv_case = 1600 / 1000 * fx2211; chk("사건 건고추 신고 단가 원/kg", uv_case, 0)
ar22 = duckdb.connect().execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)").sql("SELECT floor_won_kg, specific_won_kg FROM tr.fct_applied_rate WHERE year=2022 AND hs10='0904210000'").df().iloc[0]
chk("사건 건고추 하한 2022", ar22.floor_won_kg, 0); chk("사건 건고추 종량세 2022", ar22.specific_won_kg, 0); chk("사건 건고추 하한 아래", int(uv_case < ar22.floor_won_kg), 0)
# I장 예: 건고추(270%)와 냉동고추(27%)의 세율 차이 — 세율 DB의 2022년 무협정 실행세율
mf22 = duckdb.connect().execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)").sql("SELECT hs10, mfn FROM tr.fct_applied_rate WHERE year=2022 AND hs10 IN ('0904210000','0710807000')").df().set_index("hs10").mfn
chk("건고추 mfn 2022", mf22["0904210000"], 1); chk("냉동고추 mfn 2022", mf22["0710807000"], 1)

,hs10,코드수,수입_백만달러,하한_원kg,하한아래_금액몫,중국_하한아래,비율중위,품명
19,1201901000,1.0,6980.503,196.304,0.000,0.000,3.813,채유용과 탈지대두박(脫脂大豆粕)용
22,1201909000,1.0,2523.212,196.304,0.000,0.000,6.265,기타
23,1207400000,1.0,2152.189,1057.143,0.000,0.000,2.989,참깨
24,1207991000,1.0,640.120,1025.000,0.000,0.000,3.040,들깨
21,1201903000,1.0,611.274,196.304,0.000,0.000,7.572,콩나물용
1,0703101000,1.0,230.223,133.333,0.000,0.000,3.386,NaN
10,0904210000,1.0,186.675,2300.000,0.000,0.000,5.190,건조한 것(부수지도 잘게 부수지도 않은 것)
0,0409000000,1.0,184.085,767.078,0.000,0.000,26.483,천연꿀
4,0703209000,1.0,166.800,500.000,0.013,0.013,2.415,기타
3,0703101090,1.0,145.924,133.333,0.000,0.000,3.677,기타


하한 코드 수 31  금액 가중 하한 아래 몫: 0.001


27.0

In [5]:
# 변환비 쌍의 단가 비율 r = uv_H / (uv_L × k), 원산지×연도
CONV = {"0710807000": ("0904210000", 5.0), "0811902000": ("0813402000", 3.0), "0714104000": ("0714101000", 1.0), "0714104000b": ("0714103000", 1.0), "0714401000": ("0714409000", 1.0)}
conv_pairs = pairs[pairs.apply(lambda r: (r.hs10_low, r.hs10_high) in {("0710807000", "0904210000"), ("0811902000", "0813402000"), ("0714104000", "0714101000"), ("0714104000", "0714103000"), ("0714401000", "0714409000"), ("0710807000", "0904220000")}, axis=1)].copy()
kmap = {"0904210000": 5.0, "0904220000": 5.0, "0813402000": 3.0, "0714101000": 1.0, "0714103000": 1.0, "0714409000": 1.0}
rows = []
for r in conv_pairs.itertuples(index=False):
    x = pp[pp.pair_id == r.pair_id].groupby(["stat_cd", "year"]).agg(VH=("imp_dlr_H", "sum"), QH=("imp_wgt_H", "sum"), VL=("imp_dlr_L", "sum"), QL=("imp_wgt_L", "sum")).reset_index()
    x = x[(x.QH > 0) & (x.QL > 0)]; x["r"] = (x.VH / x.QH) / ((x.VL / x.QL) * kmap[r.hs10_high]); x["pair_id"] = r.pair_id; x["high"] = r.hs10_high; x["low"] = r.hs10_low
    rows.append(x)
rr = pd.concat(rows)
t9 = rr[rr.stat_cd == "CN"].pivot_table(index="year", columns="pair_id", values="r").round(2)
display(t9)
c = rr[(rr.stat_cd == "CN") & (rr.high == "0904210000") & (rr.low == "0710807000")].set_index("year").r
chk("r 건고추/냉동고추 중국 2012", c.get(2012, np.nan), 2); chk("r 건고추/냉동고추 중국 2024", c.get(2024, np.nan), 2)
c12 = c[c.index >= 2012]; chk("r 건고추/냉동고추 중국 최소", c12.min(), 2); chk("r 건고추/냉동고추 중국 최대", c12.max(), 2)
cj = rr[(rr.stat_cd == "CN") & (rr.high == "0813402000") & (rr.low == "0811902000") & (rr.year >= 2017)].r
chk("r 건대추/냉동대추 중국 최소", cj.min(), 2); chk("r 건대추/냉동대추 중국 최대", cj.max(), 2)
# 건대추 쌍(2026-09-13): 비율 0.02~0.06은 냉동대추 물량이 0톤인 해의 표본 단가에서 나온다. 두 코드 모두 5톤 이상인 해만 따로 등록하고 중국산 냉동대추 톤·단가를 남긴다
cjt = rr[(rr.stat_cd == "CN") & (rr.high == "0813402000") & (rr.low == "0811902000") & (rr.year >= 2017) & (rr.year <= 2025)].set_index("year")
cj5 = cjt[(cjt.QH >= 5000) & (cjt.QL >= 5000)]
chk("r 건대추/냉동대추 중국 5톤 이상 최소", cj5.r.min(), 2); chk("r 건대추/냉동대추 중국 5톤 이상 최대", cj5.r.max(), 2); chk("r 건대추/냉동대추 5톤 이상 해 수", len(cj5), 0)
chk("r 건대추/냉동대추 5톤 이상 첫해", int(cj5.index.min()), 0); chk("r 건대추/냉동대추 5톤 이상 끝해", int(cj5.index.max()), 0)
cj0 = cjt[cjt.QL < 5000]; chk("냉동대추 중국 5톤 미만 해 수", len(cj0), 0); chk("냉동대추 중국 5톤 미만 해 단가 최소", (cj0.VL / cj0.QL).min(), 0); chk("냉동대추 중국 5톤 미만 해 단가 최대", (cj0.VL / cj0.QL).max(), 0); chk("r 건대추/냉동대추 5톤 미만 최대", cj0.r.max(), 2)
for y in [2018, 2019, 2020, 2021, 2022, 2024]: chk(f"냉동대추 중국 톤 {y}", cjt.QL[y] / 1000, 1); chk(f"냉동대추 중국 단가 {y}", cjt.VL[y] / cjt.QL[y], 2)
chk("건대추 중국 톤 2017~2025 최소", cjt.QH.min() / 1000, 0); chk("건대추 중국 톤 2017~2025 최대", cjt.QH.max() / 1000, 0); chk("건대추 중국 단가 2017~2025 최소", (cjt.VH / cjt.QH).min(), 2); chk("건대추 중국 단가 2017~2025 최대", (cjt.VH / cjt.QH).max(), 2)
chk("건대추 하한 원", float(pc[pc.hs10 == "0813402000"].floor_won_kg.dropna().iloc[-1]), 0)
jj_ = pp[(pp.pair_id == cjt.pair_id.iloc[0]) & (pp.year >= 2017) & (pp.year <= 2025)]; chk("건대추 중국 몫 2017~2025", jj_[jj_.stat_cd == "CN"].imp_dlr_H.sum() / jj_.imp_dlr_H.sum(), 2)

pair_id,P0078,P0079,P0080,P0460
year,,,,
2007,NaN,NaN,0.60,NaN
2008,NaN,NaN,0.58,NaN
2011,NaN,NaN,1.07,NaN
2012,1.10,1.06,NaN,NaN
2013,0.84,0.83,NaN,NaN
2014,0.83,0.58,NaN,NaN
2015,1.02,0.68,NaN,NaN
2016,1.02,0.98,NaN,NaN
2017,1.01,1.10,0.02,NaN


0.96

In [6]:
# 3c 미러 통계(scripts/21·22의 산출을 읽어 본문 수치를 등록한다. 갭 회귀 자체는 22가 정본)
mg = pd.read_csv(os.path.join(O, "미러_갭_회귀.csv")); mp = pd.read_csv(os.path.join(O, "미러_갭_패널.csv"), dtype={"grp": str})
gg = pd.read_csv(os.path.join(O, "미러_건조생강.csv"), dtype={"hs6": str}); tg = pd.read_csv(os.path.join(O, "미러_단가갭_HS6원산지.csv"), dtype={"hs6": str})
display(mg); display(gg[["hs6", "year", "uv_x", "uv_m", "gap_uv"]].round(2))
chk("미러 패널 관측", len(mp), 0); chk("미러 집단 수", mp.grp.nunique(), 0); chk("미러 보고국 수", mp.reporter.nunique(), 0)
for row in mg.itertuples(index=False): chk(f"미러 {row.dep} {row.var}", row.coef, 4); chk(f"미러 {row.dep} {row.var} t", row.t, 2)
g12 = gg[gg.hs6 == "091012"].set_index("year")
for y in [2017, 2020, 2021, 2022, 2023, 2024]: chk(f"건조생강 미러 단가 갭 {y}", float(g12.gap_uv.get(y, np.nan)), 2)
chk("건조생강 중국 보고 단가 2020", float(g12.uv_x.get(2020, np.nan)), 2); chk("건조생강 한국 단가 2020", float(g12.uv_m.get(2020, np.nan)), 2)
ta = tg[tg.agri]; chk("농식품 단가 갭 중위", ta.gap_uv.median(), 2); chk("농식품 단가 갭 90분위", ta.gap_uv.quantile(.9), 2)
chk("농식품 HS6×보고국 수", len(ta), 0)
for i, v in enumerate(ta.sort_values("gap_uv", ascending=False).gap_uv.head(5).tolist(), 1): chk(f"농식품 단가 갭 상위 {i}", v, 2)
for row in mg.itertuples(index=False): chk(f"미러 {row.dep} {row.var} 관측", row.n, 0); chk(f"미러 {row.dep} {row.var} 군집", row.clusters, 0)
chk("미러 갭 중위 금액", mp.gap_v.median(), 2); chk("미러 갭 중위 물량", mp.gap_q.median(), 2); chk("미러 갭 중위 단가", mp.gap_uv.median(), 2)
for y in range(2017, 2025):
    if y in g12.index: chk(f"건조생강 미러 단가 갭 {y}", float(g12.gap_uv[y]), 2); chk(f"건조생강 중국 보고 단가 {y}", float(g12.uv_x[y]), 2); chk(f"건조생강 한국 단가 {y}", float(g12.uv_m[y]), 2); chk(f"건조생강 중국 보고 톤 {y}", float(g12.x_kg[y]) / 1000, 0); chk(f"건조생강 한국 톤 {y}", float(g12.m_kg[y]) / 1000, 0)
g11 = gg[gg.hs6 == "091011"]
chk("신선생강 미러 단가 갭 최소", g11.gap_uv.min(), 2); chk("신선생강 미러 단가 갭 최대", g11.gap_uv.max(), 2); chk("신선생강 중국 보고 단가 최소", g11.uv_x.min(), 2); chk("신선생강 중국 보고 단가 최대", g11.uv_x.max(), 2); chk("신선생강 한국 단가 최소", g11.uv_m.min(), 2); chk("신선생강 한국 단가 최대", g11.uv_m.max(), 2)

,dep,var,coef,se,t,n,clusters
0,gap_v,mfn_mean,-0.0007,0.0005,-1.3990,21060,154
1,gap_v,mfn_gap,-0.0002,0.0005,-0.4298,21060,154
2,gap_v(중국),mfn_mean,-0.0001,0.0005,-0.2006,1552,138
3,gap_v(중국),mfn_gap,0.0004,0.0004,0.8334,1552,138
4,gap_q,mfn_mean,-0.0004,0.0007,-0.5755,20359,154
5,gap_q,mfn_gap,-0.0009,0.0005,-2.0459,20359,154
6,gap_q(중국),mfn_mean,0.0003,0.0006,0.4351,1518,137
7,gap_q(중국),mfn_gap,-0.0007,0.0005,-1.2081,1518,137
8,gap_uv,mfn_mean,-0.0001,0.0003,-0.5359,20359,154
9,gap_uv,mfn_gap,0.0005,0.0003,1.6457,20359,154


,hs6,year,uv_x,uv_m,gap_uv
0,091011,2017,0.63,0.63,0.00
1,091011,2018,0.75,0.84,-0.11
2,091011,2019,0.80,0.84,-0.05
3,091011,2020,1.18,1.22,-0.04
4,091011,2021,1.39,1.40,-0.00
5,091011,2022,0.47,0.58,-0.21
6,091011,2023,1.73,1.71,0.01
7,091011,2024,1.36,1.30,0.04
8,091012,2017,2.72,0.64,1.44
9,091012,2018,3.48,0.71,1.59


1.71

In [7]:
# 3e 미러 단가 갭 후보의 통관 대조(2026-09-13): 농식품 단가 갭 상위 5(HS6×보고국)의 10단위 코드별 연 단가(그 원산지)·다른 원산지 단가·미러의 금액·물량 갭·적용 세율·하한·집행 지정
mh = pd.read_csv(os.path.join(O, "미러_갭_HS6_2017_2024.csv"), dtype={"hs6": str})
top5 = tg[tg.agri].sort_values("gap_uv", ascending=False).head(5)
con5 = duckdb.connect(); con5.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)"); con5.execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)")
ar24 = con5.sql("SELECT hs10, mfn, floor_won_kg, applied_vn, applied_cn, applied_asean FROM tr.fct_applied_rate WHERE year=2024").df().set_index("hs10")
dsg = dist.groupby("hs10").agg(start=("start", "min"), end=("end", "max"))
rows, summ = [], []
for c in top5.itertuples(index=False):
    q = con5.sql(f"SELECT hs10, stat_cd, yyyymm//100 AS yr, sum(imp_dlr) v, sum(imp_wgt) w FROM s.fact_trade WHERE substr(hs10,1,6)='{c.hs6}' AND yyyymm BETWEEN 201701 AND 202512 AND imp_wgt>0 AND imp_dlr>0 GROUP BY 1,2,3").df()
    g = mh[(mh.reporter == c.reporter) & (mh.hs6 == c.hs6)].set_index("year"); g5 = g[g.index <= 2021]
    for h, qq in q.groupby("hs10"):
        tot = qq.groupby("yr").agg(v=("v", "sum"), w=("w", "sum")); og = qq[qq.stat_cd == c.reporter].groupby("yr").agg(v=("v", "sum"), w=("w", "sum")); oth = qq[qq.stat_cd != c.reporter].groupby("yr").agg(v=("v", "sum"), w=("w", "sum"))
        for y in og.index:
            rows.append(dict(reporter=c.reporter, hs6=c.hs6, hs10=h, name=names.get(h, ""), year=int(y), ton=og.w[y] / 1000, musd=og.v[y] / 1e6, uv=og.v[y] / og.w[y], uv_others=(oth.v[y] / oth.w[y]) if y in oth.index and oth.w[y] > 0 else np.nan,
                             share_origin=og.v[y] / tot.v[y], gap_v=g.gap_v.get(y, np.nan), gap_q=g.gap_q.get(y, np.nan), gap_uv=g.gap_uv.get(y, np.nan), x_ton=g.x_kg.get(y, np.nan) / 1000, m_ton=g.m_kg.get(y, np.nan) / 1000))
    d = pd.DataFrame([r for r in rows if r["reporter"] == c.reporter and r["hs6"] == c.hs6])
    h0 = d.groupby("hs10").musd.sum().idxmax(); x = d[d.hs10 == h0]; xb = x[x.musd > 0.5]
    a = ar24.loc[h0] if h0 in ar24.index else None
    summ.append(dict(reporter=c.reporter, hs6=c.hs6, gap_uv_2017_2024=c.gap_uv, gap_v_to2021=g5.gap_v.mean(), gap_q_to2021=g5.gap_q.mean(), gap_uv_to2021=g5.gap_uv.mean(), x_over_m_ton=g5.x_kg.sum() / g5.m_kg.sum(),
                     hs10=h0, name=names.get(h0, ""), share_of_hs6=x.musd.sum() / d.musd.sum(), uv_min=x.uv.min(), uv_max=x.uv.max(), uv_slope=np.polyfit(xb.year, np.log(xb.uv), 1)[0] if len(xb) >= 4 else np.nan, ratio_others_median=x.ratio_to_others.median() if "ratio_to_others" in x else (x.uv / x.uv_others).median(),
                     mfn=a.mfn if a is not None else np.nan, applied_origin=(a.applied_vn if c.reporter == "VN" else a.applied_cn) if a is not None else np.nan, floor_won_kg=a.floor_won_kg if a is not None else np.nan,
                     pre_review=h0 in set(pre.hs10), dist_start=dsg.start.get(h0, ""), dist_end=dsg.end.get(h0, "")))
cand = pd.DataFrame(rows); cand["ratio_to_others"] = cand.uv / cand.uv_others
cand.to_csv(os.path.join(O, "미러_후보_통관대조.csv"), index=False, encoding="utf-8-sig")
t7b = pd.DataFrame(summ); t7b.to_csv(os.path.join(O, "미러_후보_통관대조_요약.csv"), index=False, encoding="utf-8-sig"); display(t7b.round(2))
for r_ in t7b.itertuples(index=False):
    k = f"{r_.reporter} {r_.hs6}"
    chk(f"후보 {k} 금액 갭", r_.gap_v_to2021, 2); chk(f"후보 {k} 물량 갭", r_.gap_q_to2021, 2); chk(f"후보 {k} 단가 갭", r_.gap_uv_to2021, 2); chk(f"후보 {k} 상대국 톤/한국 톤", r_.x_over_m_ton, 2)
    chk(f"후보 {k} 대표 코드 몫", r_.share_of_hs6, 2); chk(f"후보 {k} 단가 최소", r_.uv_min, 2); chk(f"후보 {k} 단가 최대", r_.uv_max, 2); chk(f"후보 {k} 단가 기울기", r_.uv_slope, 3); chk(f"후보 {k} 다른 원산지 대비 중위", r_.ratio_others_median, 2)
    chk(f"후보 {k} mfn", r_.mfn, 1); chk(f"후보 {k} 적용 세율", r_.applied_origin, 1); chk(f"후보 {k} 하한 있음", int(pd.notna(r_.floor_won_kg)), 0); chk(f"후보 {k} 유통이력", int(bool(r_.dist_start)), 0); chk(f"후보 {k} 사전세액심사", int(r_.pre_review), 0)
chk("후보 금액 갭 절대치 최대(베트남)", t7b[t7b.reporter == "VN"].gap_v_to2021.abs().max(), 2); chk("후보 물량 갭 최소", t7b.gap_q_to2021.min(), 2); chk("후보 물량 갭 최대", t7b.gap_q_to2021.max(), 2)
# 2022년 이후 베트남 보고 금액이 한국 수입액을 넘어선다(경유·재수출)
v22 = mh[(mh.reporter == "VN") & mh.hs6.isin(top5[top5.reporter == "VN"].hs6) & (mh.year >= 2022)]; chk("후보 베트남 2022~ 금액 갭 최소", v22.gap_v.min(), 2); chk("후보 베트남 2022~ 금액 갭 최대", v22.gap_v.max(), 2)
# 견과 조제품: 2022년 볶은 참깨 분리 뒤 기타 코드의 베트남 단가
s19 = cand[(cand.reporter == "VN") & (cand.hs10 == "2008199000")].set_index("year"); chk("후보 견과 기타 베트남 단가 2021", s19.uv[2021], 2); chk("후보 견과 기타 베트남 단가 2025", s19.uv[2025], 2)
s193 = cand[(cand.reporter == "VN") & (cand.hs10 == "2008193000")]; chk("후보 볶은 참깨 베트남 천 톤 최소", s193.ton.min() / 1000, 0); chk("후보 볶은 참깨 베트남 천 톤 최대", s193.ton.max() / 1000, 0); chk("후보 볶은 참깨 베트남 단가 최소", s193.uv.min(), 2); chk("후보 볶은 참깨 베트남 단가 최대", s193.uv.max(), 2)

,reporter,hs6,gap_uv_2017_2024,gap_v_to2021,gap_q_to2021,gap_uv_to2021,x_over_m_ton,hs10,name,share_of_hs6,uv_min,uv_max,uv_slope,ratio_others_median,mfn,applied_origin,floor_won_kg,pre_review,dist_start,dist_end
0,VN,030695,2.52,-0.06,-2.60,2.54,0.07,0306959030,염장이나 염수장한 것,0.99,0.70,0.91,0.01,0.67,32.0,0.0,NaN,False,2017-02-01,2029-04-30
1,VN,230990,2.37,-0.01,-2.43,2.41,0.09,2309901040,축우용,0.64,0.12,0.15,0.02,0.42,4.2,0.0,NaN,False,,
2,VN,210690,1.49,-0.22,-1.76,1.54,0.18,2106909099,기타,0.85,1.07,2.33,0.10,0.07,8.0,0.0,NaN,False,,
3,VN,200819,1.41,-0.03,-1.57,1.54,0.22,2008199000,기타,0.58,1.31,8.23,0.24,0.91,45.0,0.0,NaN,False,2017-02-01,2022-07-31
4,CN,051199,1.37,-2.02,-3.41,1.39,0.03,0511999040,원피의 페어링(paring)과 이와 유사한 웨이스트(waste),0.64,0.34,0.56,0.06,0.89,8.0,0.0,NaN,False,,


2.18

In [8]:
# 3d 파일럿(고추·생강)의 인용 수치: outputs/pilot_series.csv에서 계산해 등록한다. 계산 근거는 파일럿_고추_생강.md
ps = pd.read_csv(os.path.join(O, "pilot_series.csv"), dtype={"hs10": str})
def ann(g, cc="CN"):
    x = ps[(ps.grp == g) & (ps.stat_cd == cc)].groupby("yr").agg(v=("imp_dlr", "sum"), w=("imp_wgt", "sum")); x = x[x.w > 0]; x["uv"] = x.v / x.w; x["ton"] = x.w / 1000; return x
def mon(g, cc="CN"):
    x = ps[(ps.grp == g) & (ps.stat_cd == cc)].groupby("yyyymm").agg(v=("imp_dlr", "sum"), w=("imp_wgt", "sum")); x["uv"] = np.where(x.w > 0, x.v / x.w.replace(0, np.nan), np.nan); x["ton"] = x.w / 1000; return x
gd, gf = ann("생강건조"), ann("생강신선")
for y in [2012, 2020, 2021]: chk(f"파일럿 건조생강 중국 단가 {y}", gd.uv[y], 2); chk(f"파일럿 건조/신선×10 {y}", gd.uv[y] / (gf.uv[y] * 10), 2)
gm = mon("생강건조"); chk("파일럿 건조생강 중국 2021.11 톤", float(gm.ton.get(202111, 0)), 0)
g22 = gm[(gm.index >= 202201)]; chk("파일럿 건조생강 중국 2022~ 월 톤 최대", g22.ton.max(), 0)
gda = gd.loc[2022:2025]; chk("파일럿 건조생강 중국 연 단가 2022~2025 최소", gda.uv.min(), 2); chk("파일럿 건조생강 중국 연 단가 2022~2025 최대", gda.uv.max(), 2)
pe = mon("생강건조", "PE"); pe = pe[(pe.index >= 202111) & (pe.index <= 202209) & (pe.w > 0)]
chk("파일럿 건조생강 페루 월 톤 최소", pe.ton.min(), 0); chk("파일럿 건조생강 페루 월 톤 최대", pe.ton.max(), 0); chk("파일럿 건조생강 페루 단가 최소", pe.uv.min(), 2); chk("파일럿 건조생강 페루 단가 최대", pe.uv.max(), 2)
vn = ann("생강건조", "VN"); chk("파일럿 건조생강 베트남 단가 2022", float(vn.uv.get(2022, np.nan)), 2)
fr, dr = ann("냉동고추"), ann("건고추"); yy = [y for y in range(2007, 2026) if y in fr.index and y in dr.index]
chk("파일럿 냉동고추 중국 연 톤 최소", fr.ton.loc[yy].min(), 0); chk("파일럿 냉동고추 중국 연 톤 최대", fr.ton.loc[yy].max(), 0); chk("파일럿 건고추 중국 연 톤 최소", dr.ton.loc[yy].min(), 0); chk("파일럿 건고추 중국 연 톤 최대", dr.ton.loc[yy].max(), 0)
chk("파일럿 냉동/건조 톤 배수 최소", (fr.ton.loc[yy] / dr.ton.loc[yy]).min(), 0); chk("파일럿 냉동/건조 톤 배수 최대", (fr.ton.loc[yy] / dr.ton.loc[yy]).max(), 0)
chk("파일럿 냉동×5/건조 최소", (5 * fr.uv.loc[yy] / dr.uv.loc[yy]).min(), 2); chk("파일럿 냉동×5/건조 최대", (5 * fr.uv.loc[yy] / dr.uv.loc[yy]).max(), 2)
d35 = dr.loc[2023:2025]; chk("파일럿 건고추 중국 단가 2023~2025 최소", d35.uv.min(), 2); chk("파일럿 건고추 중국 단가 2023~2025 최대", d35.uv.max(), 2)
fr_ = pd.Series({y: 2300 / (d35.uv[y] * fxy[y]) for y in d35.index}); chk("파일럿 건고추 하한/단가 최소", fr_.min(), 2); chk("파일럿 건고추 하한/단가 최대", fr_.max(), 2)
gr_ = pd.Series({y: 247 / (gd.uv[y] * fxy[y]) for y in range(2022, 2026) if y in gd.index}); chk("파일럿 생강 하한/건조 단가 최소", gr_.min(), 2); chk("파일럿 생강 하한/건조 단가 최대", gr_.max(), 2)
gr2 = pd.Series({y: 247 / (gf.uv[y] * 10 * fxy[y]) for y in range(2012, 2022) if y in gf.index}); chk("파일럿 생강 하한/신선×10 최소", gr2.min(), 2); chk("파일럿 생강 하한/신선×10 최대", gr2.max(), 2)

0.05

### §4 집행 지정의 사건연구 — 표 4(사전세액심사·유통이력·건조생강 두 사건일), 표 5(코드별 단절), 표 6(지정 전 과소신고 유무 분할)

In [9]:
def event_study(series, treated, events, controls, y="lnq", win=24, min_post=6):
    """series: (hs10, stat_cd, yyyymm, y). treated: dict hs10->event yyyymm. controls: hs10 목록(사건 없음).
    처리 코드마다 자기 사건월을, 대조 코드는 같은 창을 처리 코호트별로 쌓는다(stacked)."""
    def mdiff(a, b): return (a // 100 - b // 100) * 12 + (a % 100 - b % 100)
    stacks = []
    for h, m in treated.items():
        ctrl = controls[h] if isinstance(controls, dict) else controls
        s = series[series.hs10.isin([h] + list(ctrl))].copy()
        s["j"] = s.yyyymm.map(lambda x: mdiff(x, m)); s = s[(s.j >= -win) & (s.j <= win)]
        s["tr"] = (s.hs10 == h).astype(int); s["stk"] = h; stacks.append(s)
    d = pd.concat(stacks); d = d[d.groupby(["stk", "hs10", "stat_cd"]).j.transform("size") >= 24]
    # 대조 코드는 처리 코드가 있는 원산지에서만, 그리고 창 안 수입액 상위 100개까지만 쓴다
    tro = set(d[d.tr == 1].stat_cd); d = d[d.stat_cd.isin(tro)]
    top = d[d.tr == 0].groupby("hs10").v.sum().sort_values(ascending=False).head(100).index
    d = d[(d.tr == 1) | d.hs10.isin(top)]
    d["ic"] = d.stk + "|" + d.hs10 + "|" + d.stat_cd; d["ct"] = d.stk + "|" + d.stat_cd + "|" + d.yyyymm.astype(str)
    js = [j for j in range(-win, win + 1) if j != -1]
    for j in js: d[f"D{j}"] = ((d.j == j) & (d.tr == 1)).astype(float)
    xs = [f"D{j}" for j in js if d[f"D{j}"].sum() > 0]
    r = fe_ols(d, y, xs, ["ic", "ct"], "hs10"); r["j"] = [int(c[1:]) for c in r.index]
    return r.reset_index(drop=True), d
def monthly(codes):
    con2 = duckdb.connect(); con2.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)")
    q = con2.sql(f"""SELECT hs10, stat_cd, yyyymm, sum(imp_dlr) v, sum(imp_wgt) w FROM s.fact_trade WHERE hs10 IN ({",".join("'"+c+"'" for c in codes)}) AND imp_wgt>0 AND imp_dlr>0 GROUP BY 1,2,3""").df()
    q["lnq"] = np.log(q.w); q["lnuv"] = np.log(q.v / q.w); q["lnv"] = np.log(q.v); return q
def summarize(r, label):
    pre = r[(r.j < -1) & (r.j >= -12)]; post = r[(r.j >= 3) & (r.j <= 12)]
    out = dict(label=label, pre_mean=pre.coef.mean(), pre_absmax_t=pre.t.abs().max(), post_mean=post.coef.mean(), post_mean_t=post.coef.mean() / np.sqrt((post.se ** 2).mean() / len(post)) if len(post) else np.nan, n=int(r.n.iloc[0]), clusters=int(r.clusters.iloc[0]))
    return out
pair_codes = set(pairs.hs10_high) | set(pairs.hs10_low)
treated_all = set(treat[treat.type.isin(["감시형", "감시형(혼합 전신)", "세율형"])].hs10) | set(treat.mate_hs10.dropna())
designated = set(pre.hs10) | set(dist.hs10)
controls = sorted(pair_codes - treated_all - designated)
print("대조 코드", len(controls))

대조 코드 212


In [10]:
# 4b 집행 지정: 사전세액심사(원산지 중국 단가·물량), 유통이력(지정 코드 물량). 파동별 지정일, 24개월 창
pre_ev = pre.assign(m=pre.aplyStrtDt.str[:7].str.replace("-", "").astype(int)).groupby("hs10").m.min().to_dict()
pre_ev = {h: m for h, m in pre_ev.items() if m >= 201401}   # 2016년 이전 표시분(2007 파동)은 창 밖
dist_ev = dist.assign(m=dist.start.str[:7].str.replace("-", "").astype(int)).groupby("hs10").m.min().to_dict()
dist_ev = {h: m for h, m in dist_ev.items() if m >= 200901}
ser2 = monthly(sorted(set(pre_ev) | set(dist_ev) | set(controls)))
out = []
for name, evd, yv, orig in [("사전세액심사 단가(중국)", pre_ev, "lnuv", "CN"), ("사전세액심사 물량(중국)", pre_ev, "lnq", "CN"), ("사전세액심사 단가(전체)", pre_ev, "lnuv", None), ("유통이력 물량(전체)", dist_ev, "lnq", None), ("유통이력 물량(중국)", dist_ev, "lnq", "CN")]:
    s = ser2 if orig is None else ser2[ser2.stat_cd == orig]
    r, d = event_study(s, evd, None, [c for c in controls if c not in evd], y=yv)
    x = summarize(r, name); x["r"] = r; out.append(x)
t11 = pd.DataFrame([{k: v for k, v in x.items() if k != "r"} for x in out]).round(3); display(t11)
for x in out: chk(x["label"] + " post", x["post_mean"], 3); chk(x["label"] + " pre|t|max", x["pre_absmax_t"], 2); chk(x["label"] + " t", x["post_mean_t"], 2); chk(x["label"] + " 관측", x["n"], 0); chk(x["label"] + " 군집", x["clusters"], 0)
ES_PRE = {x["label"]: x["r"] for x in out}

,label,pre_mean,pre_absmax_t,post_mean,post_mean_t,n,clusters
0,사전세액심사 단가(중국),-0.107,2.105,-0.039,-0.932,136561,132
1,사전세액심사 물량(중국),0.408,1.563,0.089,0.531,136561,132
2,사전세액심사 단가(전체),0.061,2.113,0.050,1.555,953813,134
3,유통이력 물량(전체),0.068,1.876,0.031,0.775,5229992,216
4,유통이력 물량(중국),0.039,1.125,-0.102,-1.582,519087,188


In [11]:
# 4c 건조생강: 공식 지정일(2022-03) 대 선행 사건일(2021-11)
gin = {"0910112000": 202203, "0910122000": 202203}
res_g = {}
for lab, m in [("공식 지정 2022-03", 202203), ("선행 사건 2021-11", 202111)]:
    r, d = event_study(ser2[ser2.stat_cd == "CN"], {h: m for h in gin}, None, [c for c in controls], y="lnuv")
    res_g[lab] = summarize(r, "건조생강 단가 " + lab)
t12 = pd.DataFrame(res_g).T.round(3); display(t12)
for lab, x in res_g.items(): chk(x["label"] + " post", x["post_mean"], 3); chk(x["label"] + " pre|t|max", x["pre_absmax_t"], 2); chk(x["label"] + " t", x["post_mean_t"], 2); chk(x["label"] + " 관측", x["n"], 0); chk(x["label"] + " 군집", x["clusters"], 0)

,label,pre_mean,pre_absmax_t,post_mean,post_mean_t,n,clusters
공식 지정 2022-03,건조생강 단가 공식 지정 2022-03,-1.875467,28.727797,0.039421,0.397633,9015,102
선행 사건 2021-11,건조생강 단가 선행 사건 2021-11,0.42447,3.983925,2.239263,27.262726,9021,102


In [12]:
# 4e 이질성: 사건연구 평균 뒤의 코드별 단절. 처리 코드마다 (사후 3~12개월 평균 − 사전 12개월 평균)에서 대조 코드 평균의 같은 차이를 뺀 값
def code_breaks(series_all, evd, y, controls_):
    def md(a, b): return (a // 100 - b // 100) * 12 + (a % 100 - b % 100)
    out = []
    for h, m in evd.items():
        t = series_all[series_all.hs10 == h].copy(); t["j"] = t.yyyymm.map(lambda x: md(x, m))
        pre_ = t[(t.j >= -12) & (t.j <= -1)]; post = t[(t.j >= 3) & (t.j <= 12)]
        if len(pre_) < 6 or len(post) < 6: continue
        c = series_all[series_all.hs10.isin(controls_)].copy(); c["j"] = c.yyyymm.map(lambda x: md(x, m))
        cc = c[(c.j >= -12) & (c.j <= -1)].groupby("hs10")[y].mean().to_frame("a").join(c[(c.j >= 3) & (c.j <= 12)].groupby("hs10")[y].mean().to_frame("b")).dropna()
        out.append(dict(hs10=h, event=m, d=(post[y].mean() - pre_[y].mean()) - (cc.b - cc.a).mean(), pre_v_musd=pre_.v.sum() / 1e6))
    return pd.DataFrame(out)
# 사전세액심사: 중국산 단가
cn = ser2[ser2.stat_cd == "CN"]
hb = code_breaks(cn, pre_ev, "lnuv", [c for c in controls if c not in pre_ev])
rt21 = duckdb.connect().execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)").sql("SELECT hs10, mfn, floor_won_kg FROM tr.fct_applied_rate WHERE year=2021").df()
hb = hb.merge(rt21, on="hs10", how="left"); hb["name"] = hb.hs10.map(pre.drop_duplicates("hs10").set_index("hs10").name_ko)
hb.to_csv(os.path.join(O, "사건연구_이질성_사전세액심사.csv"), index=False, encoding="utf-8-sig")
display(hb.sort_values("d", ascending=False).round(2).head(12))
print("중국 단가 단절 분위:", hb.d.describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_dict())
chk("사전세액심사 단가 단절 중위", hb.d.median(), 2); chk("사전세액심사 단가 단절 90분위", hb.d.quantile(.9), 2); chk("사전세액심사 단가 단절 코드 수", len(hb), 0)
chk("단가 단절 건조생강 부순 것", float(hb.set_index("hs10").d.get("0910122000", np.nan)), 2); chk("단가 단절 건조생강", float(hb.set_index("hs10").d.get("0910112000", np.nan)), 2)
for row in hb.itertuples(index=False):
    chk(f"단절 {row.hs10} d", row.d, 2); chk(f"단절 {row.hs10} mfn", row.mfn, 1); chk(f"단절 {row.hs10} 지정 전 수입", row.pre_v_musd, 1)
    if pd.notna(row.floor_won_kg): chk(f"단절 {row.hs10} 하한", row.floor_won_kg, 0)
chk("단가 단절 mfn 상관", hb[["d", "mfn"]].corr().iloc[0, 1], 2); chk("단가 단절 하한 있음 중위", hb[hb.floor_won_kg.notna()].d.median(), 2); chk("단가 단절 하한 없음 중위", hb[hb.floor_won_kg.isna()].d.median(), 2)

,hs10,event,d,pre_v_musd,mfn,floor_won_kg,name
16,0910122000,202203,2.28,0.12,377.3,246.75,건조한 것(생강 부순 것)
15,0910112000,202203,1.33,0.02,377.3,246.75,건조한 것(생강)
18,1006301000,202003,0.97,0.02,513.0,NaN,멥쌀
8,0712200000,202210,0.93,1.76,135.0,133.33,양파(건조)
4,0704902000,202511,0.36,6.83,27.0,NaN,배추
2,0703201000,201702,0.32,1.78,360.0,500.00,껍질을 깐 것(마늘)
29,2001909060,202311,0.24,3.81,30.0,NaN,마늘(식초 조제)
13,0904220000,202311,0.20,7.86,270.0,2300.00,부수거나 잘게 부순 것(고춧가루)
10,0713319000,201702,0.10,4.24,607.5,NaN,기타(녹두·팥 등 0713.31)
25,1207991000,201702,0.09,34.35,40.0,1025.00,들깨


중국 단가 단절 분위: {'count': 31.0, 'mean': 0.09, 'std': 0.6, 'min': -0.83, '10%': -0.45, '25%': -0.16, '50%': -0.01, '75%': 0.15, '90%': 0.93, 'max': 2.28}


-0.02

In [13]:
def mdiff(a, b): return (a // 100 - b // 100) * 12 + (a % 100 - b % 100)
# 4g 사전세액심사 처리군을 지정 전 과소신고 크기로 분할(2026-09-13). 원가 대리: 변환비 코드(건조생강 = 신선×10, 건고추·고춧가루 = 냉동×5)는 지정 전 12개월 중국산 통관 단가,
# 그 밖은 미러 통계의 중국 보고 단가(집단×연도 단가 갭 gap_uv, 지정 전 3년 평균; 집단 라벨은 scripts/22와 같은 연계표 연결 성분). 갭 ≥ 0.5(로그)이면 "과소신고 있음".
cm_ = pd.read_csv(os.path.join(O, "comtrade_mirror_hs6_2012_2024.csv"), dtype={"cmdCode": str})
conc = duckdb.connect().execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)").sql("SELECT hs2022, hs_past FROM s.dim_hs6_concordance").df()
scope = set(cm_.cmdCode); sub = conc[conc.hs2022.isin(scope) | conc.hs_past.isin(scope)]
par = {}
def find(x):
    while par.setdefault(x, x) != x:
        par[x] = par[par[x]]; x = par[x]
    return x
for a_, b_ in zip(sub.hs2022, sub.hs_past): par[find("N" + a_)] = find("P" + b_)
grp_name = {}
for n_ in list(par): grp_name.setdefault(find(n_), set()).add(n_[1:])
label = {r_: min(v_) for r_, v_ in grp_name.items()}
def grp_of(hs6): return label.get(find("N" + hs6), hs6) if ("N" + hs6) in par else hs6
mcn = mp[mp.reporter == "CN"]
CONV_PRE = {"0910112000": ("0910111000", 10.0), "0910122000": ("0910111000", 10.0), "0904210000": ("0710807000", 5.0), "0904220000": ("0710807000", 5.0)}
rows = []
for row in hb.itertuples(index=False):
    m = int(row.event); ey = m // 100; g = grp_of(row.hs10[:6])
    if row.hs10 in CONV_PRE:
        base, k = CONV_PRE[row.hs10]
        t_ = cn[cn.hs10 == row.hs10].copy(); t_["j"] = t_.yyyymm.map(lambda x: mdiff(x, m)); t_ = t_[(t_.j >= -12) & (t_.j <= -1)]
        b_ = cn[cn.hs10 == base].copy(); b_["j"] = b_.yyyymm.map(lambda x: mdiff(x, m)); b_ = b_[(b_.j >= -12) & (b_.j <= -1)]
        gap = np.log(k * (b_.v.sum() / b_.w.sum()) / (t_.v.sum() / t_.w.sum())) if len(t_) and len(b_) and t_.w.sum() > 0 else np.nan
        src = f"변환비 {base}x{k:g}"; ny = len(t_)
    else:
        x_ = mcn[(mcn.grp == g) & mcn.year.between(ey - 3, ey - 1)]
        gap = x_.gap_uv.mean() if len(x_) else np.nan; src = f"미러 {g} {ey-3}~{ey-1}"; ny = len(x_)
    rows.append(dict(hs10=row.hs10, name=row.name, event=m, d=row.d, pre_gap=gap, gap_src=src, gap_n=ny, mfn=row.mfn, floor_won_kg=row.floor_won_kg, pre_v_musd=row.pre_v_musd))
sp = pd.DataFrame(rows)
sp["group"] = np.where(sp.pre_gap.isna(), "갭 없음", np.where(sp.pre_gap >= 0.5, "과소신고 있음", "과소신고 없음"))
sp.to_csv(os.path.join(O, "사건연구_사전세액심사_분할_코드.csv"), index=False, encoding="utf-8-sig")
display(sp.sort_values("pre_gap", ascending=False).round(2))
for gname, n_ in sp.group.value_counts().items(): chk(f"분할 {gname} 코드 수", n_, 0)
for row in sp[sp.pre_gap.notna()].itertuples(index=False): chk(f"분할 갭 {row.hs10}", row.pre_gap, 2)
chk("분할 갭 있음 중위 단절", sp[sp.group == "과소신고 있음"].d.median(), 2); chk("분할 갭 없음 중위 단절", sp[sp.group == "과소신고 없음"].d.median(), 2)
yes = sp[sp.group == "과소신고 있음"].hs10.tolist(); no = sp[sp.group == "과소신고 없음"].hs10.tolist(); gn = sp[sp.group == "갭 없음"].hs10.tolist()
yes3 = sp[sp.pre_gap >= 0.3].hs10.tolist(); no3 = sp[sp.pre_gap.notna() & (sp.pre_gap < 0.3)].hs10.tolist()
lead = lambda hs: {h: (202111 if h in ("0910112000", "0910122000") else pre_ev[h]) for h in hs}
runs = [("있음 공식 지정일", {h: pre_ev[h] for h in yes}, "lnuv"), ("있음 선행 사건일", lead(yes), "lnuv"), ("없음", {h: pre_ev[h] for h in no}, "lnuv"), ("갭 없음", {h: pre_ev[h] for h in gn}, "lnuv"),
        ("있음(0.3) 공식 지정일", {h: pre_ev[h] for h in yes3}, "lnuv"), ("있음(0.3) 선행 사건일", lead(yes3), "lnuv"), ("없음(0.3)", {h: pre_ev[h] for h in no3}, "lnuv"),
        ("있음 물량 선행 사건일", lead(yes), "lnq"), ("없음 물량", {h: pre_ev[h] for h in no}, "lnq")]
out = []
for lab, evd, yv in runs:
    if not evd: continue
    r_, _ = event_study(cn, evd, None, [c for c in controls if c not in evd], y=yv); x_ = summarize(r_, lab); x_["n_codes"] = len(evd); out.append(x_)
t16 = pd.DataFrame(out).round(3); display(t16); t16.to_csv(os.path.join(O, "사건연구_사전세액심사_분할.csv"), index=False, encoding="utf-8-sig")
for x_ in out: chk("분할 " + x_["label"] + " post", x_["post_mean"], 3); chk("분할 " + x_["label"] + " t", x_["post_mean_t"], 2); chk("분할 " + x_["label"] + " pre 평균", x_["pre_mean"], 2); chk("분할 " + x_["label"] + " pre|t|max", x_["pre_absmax_t"], 2); chk("분할 " + x_["label"] + " 관측", x_["n"], 0); chk("분할 " + x_["label"] + " 군집", x_["clusters"], 0); chk("분할 " + x_["label"] + " 코드", x_["n_codes"], 0)

,hs10,name,event,d,pre_gap,gap_src,gap_n,mfn,floor_won_kg,pre_v_musd,group
16,0910122000,건조한 것(생강 부순 것),202203,2.28,3.56,변환비 0910111000x10,11,377.3,246.75,0.12,과소신고 있음
15,0910112000,건조한 것(생강),202203,1.33,3.51,변환비 0910111000x10,6,377.3,246.75,0.02,과소신고 있음
29,2001909060,마늘(식초 조제),202311,0.24,0.54,미러 200190 2020~2022,3,30.0,NaN,3.81,과소신고 있음
14,0910111000,신선한 것이나 냉장한 것(생강),201702,0.08,0.40,미러 091010 2014~2016,3,377.3,246.75,5.72,과소신고 없음
27,1605542091,조미오징어,201702,-0.50,0.36,미러 030722 2014~2016,3,20.0,NaN,15.55,과소신고 없음
3,0703209000,기타(마늘),201702,-0.83,0.27,미러 070320 2014~2016,3,360.0,500.00,74.07,과소신고 없음
2,0703201000,껍질을 깐 것(마늘),201702,0.32,0.27,미러 070320 2014~2016,3,360.0,500.00,1.78,과소신고 없음
28,1904901010,찌거나 삶은 쌀,201702,0.06,0.26,미러 190490 2014~2016,3,50.0,NaN,1.64,과소신고 없음
6,0710802000,마늘(냉동),201702,-0.17,0.22,미러 071080 2014~2016,3,27.0,NaN,52.95,과소신고 없음
7,0710807000,고추류(냉동),201702,-0.02,0.22,미러 071080 2014~2016,3,27.0,NaN,100.70,과소신고 없음


,label,pre_mean,pre_absmax_t,post_mean,post_mean_t,n,clusters,n_codes
0,있음 공식 지정일,-1.019,2.482,0.104,1.394,13164,103,3
1,있음 선행 사건일,0.068,1.330,1.205,5.066,12954,103,3
2,없음,-0.036,1.520,-0.107,-2.103,74932,122,22
3,갭 없음,0.031,1.432,0.129,2.089,19789,106,6
4,있음(0.3) 공식 지정일,-0.587,2.222,-0.034,-0.489,19365,105,5
5,있음(0.3) 선행 사건일,-0.043,1.502,0.509,3.088,19023,105,5
6,없음(0.3),-0.032,1.359,-0.093,-1.681,68746,120,20
7,있음 물량 선행 사건일,-0.577,2.888,-2.862,-5.018,12954,103,3
8,없음 물량,0.191,1.078,0.096,0.501,74932,122,22


### §5 강건성 — 잔여 통관 창(사후 3·6개월 제외), 연평균 환율의 하한 아래 몫

In [14]:
rob = {}
# 둘째 잔여 통관 창: 사전세액심사 단가(중국)에서 사후 0~2, 0~5개월을 뺀 평균
r0 = ES_PRE["사전세액심사 단가(중국)"]
rob["사전세액심사 단가 post 3~12"] = pd.Series(dict(coef=r0[(r0.j >= 3) & (r0.j <= 12)].coef.mean()))
rob["사전세액심사 단가 post 6~12"] = pd.Series(dict(coef=r0[(r0.j >= 6) & (r0.j <= 12)].coef.mean()))
# 일곱째 환율: 연평균 환율로 하한 아래 몫
fl2 = fl.copy(); fl2["uv_won_y"] = fl2.uv * fl2.year.map(fxy); fl2["below_y"] = fl2.uv_won_y < fl2.floor_won_kg
rob["하한 아래 몫 월환율"] = pd.Series(dict(coef=np.average(fl[fl.year >= 2012].below, weights=fl[fl.year >= 2012].imp_dlr)))
rob["하한 아래 몫 연환율"] = pd.Series(dict(coef=np.average(fl2[fl2.year >= 2012].below_y, weights=fl2[fl2.year >= 2012].imp_dlr)))
t14 = pd.DataFrame(rob).T; display(t14.round(4))
chk("잔여 창 사전세액심사 단가 3~12", rob["사전세액심사 단가 post 3~12"]["coef"], 3); chk("잔여 창 사전세액심사 단가 6~12", rob["사전세액심사 단가 post 6~12"]["coef"], 3)
chk("하한 아래 몫 월환율", rob["하한 아래 몫 월환율"]["coef"], 4); chk("하한 아래 몫 연환율", rob["하한 아래 몫 연환율"]["coef"], 4)

,coef
사전세액심사 단가 post 3~12,-0.0389
사전세액심사 단가 post 6~12,-0.0453
하한 아래 몫 월환율,0.0008
하한 아래 몫 연환율,0.0008


0.0008

### §6 검증 — 논문 A의 인용 수치 대조

In [15]:
EXPECT_A = {
 # II장 자료
 "자료 일곱 상대 대조 행": (22164, 0), "자료 일곱 상대 대조 어긋남": (0, 0), "자료 하한 있는 행": (1092, 0), "자료 종량 대안 코드": (92, 0),
 "자료 하한 최소 원": (88, 0.5), "자료 하한 최대 원": (34000, 500), "자료 별표 1의 가 행": (9944, 0), "자료 별표 1의 나 행": (278, 0),
 "사전세액심사 코드 수": (48, 0), "유통이력 코드 수": (155, 0), "유통이력 구간 수": (237, 0), "자료 품목 상세 코드": (285, 0),
 "자료 미러 행": (91318, 0), "자료 미러 보고국": (28, 0), "쌍 수": (444, 0), "패널 코드 수": (602, 0), "패널 원산지 수": (233, 0),
 "자료 두 화면 일치율 A": (99.80, 0.005), "자료 두 화면 일치율 C": (99.97, 0.005), "자료 19 일곱 상대 대조 행": (22164, 0), "자료 19 일곱 상대 어긋남": (0, 0),
 # IV.1 표 2 하한
 "하한 코드 수": (31, 0), "하한 아래 몫 전체": (0.001, 0.0005), "하한 아래 몫 연환율": (0.0008, 0.00005),
 "하한 표 1201901000 수입": (6981, 0.5), "하한 표 1201901000 하한": (196, 0.5), "하한 표 1201901000 몫": (0.000, 0.0005), "하한 표 1201901000 비율중위": (3.81, 0.005),
 "하한 표 1201909000 수입": (2523, 0.5), "하한 표 1201909000 하한": (196, 0.5), "하한 표 1201909000 몫": (0.000, 0.0005), "하한 표 1201909000 비율중위": (6.27, 0.005),
 "하한 표 1207400000 수입": (2152, 0.5), "하한 표 1207400000 하한": (1057, 0.5), "하한 표 1207400000 몫": (0.000, 0.0005), "하한 표 1207400000 비율중위": (2.99, 0.005),
 "하한 표 1207991000 수입": (640, 0.5), "하한 표 1207991000 하한": (1025, 0.5), "하한 표 1207991000 몫": (0.000, 0.0005), "하한 표 1207991000 비율중위": (3.04, 0.005),
 "하한 표 0904210000 수입": (187, 0.5), "하한 표 0904210000 하한": (2300, 0.5), "하한 표 0904210000 몫": (0.000, 0.0005), "하한 표 0904210000 비율중위": (5.19, 0.005),
 "하한 표 0409000000 수입": (184, 0.5), "하한 표 0409000000 하한": (767, 0.5), "하한 표 0409000000 몫": (0.000, 0.0005), "하한 표 0409000000 비율중위": (26.48, 0.005),
 "하한 표 0703209000 수입": (167, 0.5), "하한 표 0703209000 하한": (500, 0.5), "하한 표 0703209000 몫": (0.013, 0.0005), "하한 표 0703209000 비율중위": (2.42, 0.005),
 "r 건고추/냉동고추 중국 2012": (1.10, 0.005), "r 건고추/냉동고추 중국 2024": (0.97, 0.005), "r 건고추/냉동고추 중국 최소": (0.83, 0.005), "r 건고추/냉동고추 중국 최대": (1.10, 0.005),
 "r 건대추/냉동대추 중국 최소": (0.02, 0.005), "r 건대추/냉동대추 중국 최대": (0.88, 0.005),
 "파일럿 건고추 중국 단가 2023~2025 최소": (2.7, 0.05), "파일럿 건고추 중국 단가 2023~2025 최대": (3.6, 0.05), "파일럿 건고추 하한/단가 최소": (0.49, 0.005), "파일럿 건고추 하한/단가 최대": (0.60, 0.005),  # 논문 A: 50~65% → 49~60% 수정 대기
 # IV.1 건고추 저가 신고 사건(조심 2023관0105) — 신고 단가 1,600달러/톤을 원/kg으로 바꾼 값과 하한·종량세
 "건고추 mfn 2022": (270, 0.05), "냉동고추 mfn 2022": (27, 0.05),
 "환율 2022.11": (1364.1, 0.05), "사건 건고추 신고 단가 원/kg": (2180, 5), "사건 건고추 하한 2022": (2300, 0.5), "사건 건고추 종량세 2022": (6210, 0.5), "사건 건고추 하한 아래": (1, 0),
 # IV.2 건조생강
 "파일럿 건조생강 중국 단가 2012": (1.94, 0.005), "파일럿 건조생강 중국 단가 2020": (0.46, 0.005), "파일럿 건조/신선×10 2020": (0.04, 0.005), "파일럿 건조/신선×10 2021": (0.03, 0.005),
 "파일럿 건조생강 중국 2021.11 톤": (0, 0.5), "파일럿 건조생강 중국 2022~ 월 톤 최대": (26, 0.5), "파일럿 건조생강 중국 연 단가 2022~2025 최소": (2.16, 0.005), "파일럿 건조생강 중국 연 단가 2022~2025 최대": (2.98, 0.005),  # 논문 A: 월 2.1~2.7달러 → 연평균 2.2~3.0달러 수정 대기
 "파일럿 건조생강 페루 월 톤 최소": (51, 0.5), "파일럿 건조생강 페루 월 톤 최대": (179, 0.5), "파일럿 건조생강 페루 단가 최소": (3.05, 0.005), "파일럿 건조생강 페루 단가 최대": (4.26, 0.005), "파일럿 건조생강 베트남 단가 2022": (1.53, 0.005),  # 논문 A: 2021.11~2022.09 창, 3.1~4.3달러·베트남 1.5달러 수정 대기
 "파일럿 생강 하한/건조 단가 최소": (0.06, 0.005), "파일럿 생강 하한/건조 단가 최대": (0.09, 0.005),  # 논문 A: 7~10% → 6~9% 수정 대기
 # IV.2 표 3 미러 생강
 **{f"건조생강 미러 단가 갭 {y}": (v, 0.005) for y, v in zip(range(2017, 2025), [1.44, 1.59, 1.76, 2.40, 2.33, 0.29, -0.02, 0.15])},
 **{f"건조생강 중국 보고 단가 {y}": (v, 0.005) for y, v in zip(range(2017, 2025), [2.72, 3.48, 3.70, 4.98, 4.97, 2.24, 2.20, 2.80])},
 **{f"건조생강 한국 단가 {y}": (v, 0.005) for y, v in zip(range(2017, 2025), [0.64, 0.71, 0.64, 0.45, 0.48, 1.68, 2.24, 2.41])},
 **{f"건조생강 중국 보고 톤 {y}": (v, 0.5) for y, v in zip(range(2017, 2025), [448, 368, 294, 205, 183, 39, 46, 46])},
 **{f"건조생강 한국 톤 {y}": (v, 0.5) for y, v in zip(range(2017, 2025), [394, 392, 507, 953, 532, 40, 10, 9])},
 "신선생강 중국 보고 단가 최소": (0.47, 0.005), "신선생강 중국 보고 단가 최대": (1.73, 0.005), "신선생강 한국 단가 최소": (0.58, 0.005), "신선생강 한국 단가 최대": (1.71, 0.005), "신선생강 미러 단가 갭 최소": (-0.21, 0.005), "신선생강 미러 단가 갭 최대": (0.04, 0.005),
 # IV.3 표 4 사건연구
 "사전세액심사 단가(중국) post": (-0.039, 0.0005), "사전세액심사 단가(중국) t": (-0.93, 0.005), "사전세액심사 단가(중국) pre|t|max": (2.11, 0.005), "사전세액심사 단가(중국) 관측": (136561, 0), "사전세액심사 단가(중국) 군집": (132, 0),
 "사전세액심사 물량(중국) post": (0.089, 0.0005), "사전세액심사 물량(중국) t": (0.53, 0.005), "사전세액심사 물량(중국) pre|t|max": (1.56, 0.005), "사전세액심사 물량(중국) 관측": (136561, 0), "사전세액심사 물량(중국) 군집": (132, 0),
 "사전세액심사 단가(전체) post": (0.050, 0.0005), "사전세액심사 단가(전체) t": (1.56, 0.005), "사전세액심사 단가(전체) pre|t|max": (2.11, 0.005), "사전세액심사 단가(전체) 관측": (953813, 0), "사전세액심사 단가(전체) 군집": (134, 0),
 "유통이력 물량(전체) post": (0.031, 0.0005), "유통이력 물량(전체) t": (0.77, 0.005),  # 논문 A 표 4: 0.78 → 0.77 수정 대기 "유통이력 물량(전체) pre|t|max": (1.88, 0.005), "유통이력 물량(전체) 관측": (5229992, 0), "유통이력 물량(전체) 군집": (216, 0),
 "유통이력 물량(중국) post": (-0.102, 0.0005), "유통이력 물량(중국) t": (-1.58, 0.005), "유통이력 물량(중국) pre|t|max": (1.12, 0.005), "유통이력 물량(중국) 관측": (519087, 0), "유통이력 물량(중국) 군집": (188, 0),
 "건조생강 단가 공식 지정 2022-03 post": (0.039, 0.0005), "건조생강 단가 공식 지정 2022-03 t": (0.40, 0.005), "건조생강 단가 공식 지정 2022-03 pre|t|max": (28.73, 0.005), "건조생강 단가 공식 지정 2022-03 관측": (9015, 0), "건조생강 단가 공식 지정 2022-03 군집": (102, 0),
 "건조생강 단가 선행 사건 2021-11 post": (2.239, 0.0005), "건조생강 단가 선행 사건 2021-11 t": (27.26, 0.005), "건조생강 단가 선행 사건 2021-11 pre|t|max": (3.98, 0.005), "건조생강 단가 선행 사건 2021-11 관측": (9021, 0), "건조생강 단가 선행 사건 2021-11 군집": (102, 0),
 "잔여 창 사전세액심사 단가 3~12": (-0.039, 0.0005), "잔여 창 사전세액심사 단가 6~12": (-0.045, 0.0005),
 # IV.4 표 5 이질성 — 논문 A 표 5: 참깨 단절 −0.25→−0.26·지정 전 수입 39.6→39.5, 메밀 −0.78→−0.79 수정 대기
 "사전세액심사 단가 단절 중위": (-0.01, 0.005), "사전세액심사 단가 단절 90분위": (0.93, 0.005), "사전세액심사 단가 단절 코드 수": (31, 0), "단가 단절 mfn 상관": (0.12, 0.005), "단가 단절 하한 있음 중위": (0.06, 0.005), "단가 단절 하한 없음 중위": (-0.02, 0.005),
 **{f"단절 {h} d": (d, 0.005) for h, d in [("0910122000", 2.28), ("0910112000", 1.33), ("1006301000", 0.97), ("0712200000", 0.93), ("0704902000", 0.36), ("0703201000", 0.32), ("1207400000", -0.26), ("0713329000", -0.35), ("1202420000", -0.45), ("1605542091", -0.50), ("1008100000", -0.79), ("0703209000", -0.83)]},
 **{f"단절 {h} mfn": (m, 0.05) for h, m in [("0910122000", 377.3), ("0910112000", 377.3), ("1006301000", 513.0), ("0712200000", 135.0), ("0704902000", 27.0), ("0703201000", 360.0), ("1207400000", 630.0), ("0713329000", 420.8), ("1202420000", 230.5), ("1605542091", 20.0), ("1008100000", 256.1), ("0703209000", 360.0)]},
 **{f"단절 {h} 하한": (f, 0.5) for h, f in [("0910122000", 247), ("0910112000", 247), ("0712200000", 133), ("0703201000", 500), ("1207400000", 1057), ("0703209000", 500)]},
 **{f"단절 {h} 지정 전 수입": (v, 0.05) for h, v in [("0910122000", 0.1), ("0910112000", 0.0), ("1006301000", 0.0), ("0712200000", 1.8), ("0704902000", 6.8), ("0703201000", 1.8), ("1207400000", 39.5), ("0713329000", 29.2), ("1202420000", 0.8), ("1605542091", 15.6), ("1008100000", 0.9), ("0703209000", 74.1)]},
 # IV.5 표 6 미러 회귀
 "미러 패널 관측": (21060, 0), "미러 집단 수": (154, 0), "미러 보고국 수": (28, 0), "미러 갭 중위 금액": (0.23, 0.005), "미러 갭 중위 물량": (0.44, 0.005), "미러 갭 중위 단가": (-0.12, 0.005),
 "미러 gap_v mfn_mean": (-0.0007, 0.00005), "미러 gap_v mfn_mean t": (-1.40, 0.005), "미러 gap_v mfn_gap": (-0.0002, 0.00005), "미러 gap_v mfn_gap t": (-0.43, 0.005), "미러 gap_v mfn_mean 관측": (21060, 0), "미러 gap_v mfn_mean 군집": (154, 0),
 "미러 gap_q mfn_mean": (-0.0004, 0.00005), "미러 gap_q mfn_mean t": (-0.58, 0.005), "미러 gap_q mfn_gap": (-0.0009, 0.00005), "미러 gap_q mfn_gap t": (-2.05, 0.005), "미러 gap_q mfn_mean 관측": (20359, 0), "미러 gap_q mfn_mean 군집": (154, 0),
 "미러 gap_uv mfn_mean": (-0.0001, 0.00005), "미러 gap_uv mfn_mean t": (-0.54, 0.005), "미러 gap_uv mfn_gap": (0.0005, 0.00005), "미러 gap_uv mfn_gap t": (1.65, 0.005), "미러 gap_uv mfn_mean 관측": (20359, 0), "미러 gap_uv mfn_mean 군집": (154, 0),
 "미러 gap_uv(중국) mfn_mean": (-0.0004, 0.00005), "미러 gap_uv(중국) mfn_mean t": (-1.51, 0.005), "미러 gap_uv(중국) mfn_gap": (0.0010, 0.00005), "미러 gap_uv(중국) mfn_gap t": (2.02, 0.005), "미러 gap_uv(중국) mfn_mean 관측": (1518, 0), "미러 gap_uv(중국) mfn_mean 군집": (137, 0),
 "미러 gap_q(농식품) mfn_mean": (-0.0001, 0.00005), "미러 gap_q(농식품) mfn_mean t": (-0.11, 0.005), "미러 gap_q(농식품) mfn_gap": (-0.0010, 0.00005), "미러 gap_q(농식품) mfn_gap t": (-2.01, 0.005), "미러 gap_q(농식품) mfn_mean 관측": (11457, 0), "미러 gap_q(농식품) mfn_mean 군집": (102, 0),
 "미러 gap_uv(농식품) mfn_mean": (-0.0001, 0.00005), "미러 gap_uv(농식품) mfn_mean t": (-0.26, 0.005), "미러 gap_uv(농식품) mfn_gap": (0.0002, 0.00005), "미러 gap_uv(농식품) mfn_gap t": (1.52, 0.005), "미러 gap_uv(농식품) mfn_mean 관측": (11457, 0), "미러 gap_uv(농식품) mfn_mean 군집": (102, 0),
 "농식품 HS6×보고국 수": (399, 0), "농식품 단가 갭 중위": (-0.07, 0.005), "농식품 단가 갭 90분위": (0.30, 0.005),
 **{f"농식품 단가 갭 상위 {i}": (v, 0.005) for i, v in enumerate([2.52, 2.37, 1.49, 1.41, 1.37], 1)},
 # IV.4 분할(2026-09-13, §4g) — 논문 A IV.4 표 6 반영
 "분할 과소신고 있음 코드 수": (3, 0), "분할 과소신고 없음 코드 수": (22, 0), "분할 갭 없음 코드 수": (6, 0),
 "분할 갭 0910122000": (3.56, 0.005), "분할 갭 0910112000": (3.51, 0.005), "분할 갭 2001909060": (0.54, 0.005), "분할 갭 0910111000": (0.40, 0.005), "분할 갭 1605542091": (0.36, 0.005), "분할 갭 0904210000": (-0.02, 0.005),
 "분할 있음 공식 지정일 post": (0.104, 0.0005), "분할 있음 공식 지정일 t": (1.39, 0.005), "분할 있음 공식 지정일 pre|t|max": (2.48, 0.005), "분할 있음 공식 지정일 관측": (13164, 0), "분할 있음 공식 지정일 군집": (103, 0),
 "분할 있음 선행 사건일 post": (1.205, 0.0005), "분할 있음 선행 사건일 t": (5.07, 0.005), "분할 있음 선행 사건일 pre|t|max": (1.33, 0.005), "분할 있음 선행 사건일 관측": (12954, 0), "분할 있음 선행 사건일 군집": (103, 0),
 "분할 없음 post": (-0.107, 0.0005), "분할 없음 t": (-2.10, 0.005), "분할 없음 pre|t|max": (1.52, 0.005), "분할 없음 관측": (74932, 0), "분할 없음 군집": (122, 0),
 "분할 갭 없음 post": (0.129, 0.0005), "분할 갭 없음 t": (2.09, 0.005), "분할 갭 없음 pre|t|max": (1.43, 0.005), "분할 갭 없음 관측": (19789, 0), "분할 갭 없음 군집": (106, 0),
 "분할 있음(0.3) 공식 지정일 post": (-0.034, 0.0005), "분할 있음(0.3) 선행 사건일 post": (0.509, 0.0005), "분할 있음(0.3) 선행 사건일 t": (3.09, 0.005), "분할 없음(0.3) post": (-0.093, 0.0005), "분할 없음(0.3) t": (-1.68, 0.005), "분할 있음(0.3) 공식 지정일 코드": (5, 0),
 # IV.5 미러 후보 통관 대조(2026-09-13, §3e) — 논문 A IV.5 끝 문단 반영
 **{f"후보 {k} 금액 갭": (v, 0.005) for k, v in [("VN 030695", -0.06), ("VN 230990", -0.01), ("VN 210690", -0.22), ("VN 200819", -0.03), ("CN 051199", -2.02)]},
 **{f"후보 {k} 물량 갭": (v, 0.005) for k, v in [("VN 030695", -2.60), ("VN 230990", -2.43), ("VN 210690", -1.76), ("VN 200819", -1.57), ("CN 051199", -3.41)]},
 **{f"후보 {k} 단가 갭": (v, 0.005) for k, v in [("VN 030695", 2.54), ("VN 230990", 2.41), ("VN 210690", 1.54), ("VN 200819", 1.54), ("CN 051199", 1.39)]},
 **{f"후보 {k} 상대국 톤/한국 톤": (v, 0.005) for k, v in [("VN 030695", 0.07), ("VN 230990", 0.09), ("VN 210690", 0.18), ("VN 200819", 0.22), ("CN 051199", 0.03)]},
 **{f"후보 {k} 단가 최소": (v, 0.005) for k, v in [("VN 030695", 0.70), ("VN 230990", 0.12), ("VN 210690", 1.07), ("VN 200819", 1.31), ("CN 051199", 0.34)]},
 **{f"후보 {k} 단가 최대": (v, 0.005) for k, v in [("VN 030695", 0.91), ("VN 230990", 0.15), ("VN 210690", 2.33), ("VN 200819", 8.23), ("CN 051199", 0.56)]},
 **{f"후보 {k} 다른 원산지 대비 중위": (v, 0.005) for k, v in [("VN 030695", 0.67), ("VN 230990", 0.42), ("VN 210690", 0.07), ("VN 200819", 0.91), ("CN 051199", 0.89)]},
 **{f"후보 {k} mfn": (v, 0.05) for k, v in [("VN 030695", 32.0), ("VN 230990", 4.2), ("VN 210690", 8.0), ("VN 200819", 45.0), ("CN 051199", 8.0)]},
 **{f"후보 {k} 적용 세율": (0.0, 0.05) for k in ["VN 030695", "VN 230990", "VN 210690", "VN 200819", "CN 051199"]},
 **{f"후보 {k} 하한 있음": (0, 0) for k in ["VN 030695", "VN 230990", "VN 210690", "VN 200819", "CN 051199"]},
 **{f"후보 {k} 사전세액심사": (0, 0) for k in ["VN 030695", "VN 230990", "VN 210690", "VN 200819", "CN 051199"]},
 **{f"후보 {k} 유통이력": (v, 0) for k, v in [("VN 030695", 1), ("VN 230990", 0), ("VN 210690", 0), ("VN 200819", 1), ("CN 051199", 0)]},
 "후보 금액 갭 절대치 최대(베트남)": (0.22, 0.005), "후보 물량 갭 최소": (-3.41, 0.005), "후보 물량 갭 최대": (-1.57, 0.005), "후보 베트남 2022~ 금액 갭 최소": (0.33, 0.005), "후보 베트남 2022~ 금액 갭 최대": (0.69, 0.005),
 "후보 견과 기타 베트남 단가 2021": (1.91, 0.005), "후보 견과 기타 베트남 단가 2025": (8.23, 0.005), "후보 볶은 참깨 베트남 천 톤 최소": (27, 0.5), "후보 볶은 참깨 베트남 천 톤 최대": (35, 0.5), "후보 볶은 참깨 베트남 단가 최소": (1.81, 0.005), "후보 볶은 참깨 베트남 단가 최대": (2.18, 0.005),
 # IV.1 건대추(2026-09-13) — 논문 A IV.1·논문 B V.2 문장 반영
 "r 건대추/냉동대추 중국 5톤 이상 최소": (0.43, 0.005), "r 건대추/냉동대추 중국 5톤 이상 최대": (0.88, 0.005), "r 건대추/냉동대추 5톤 이상 해 수": (6, 0), "r 건대추/냉동대추 5톤 이상 첫해": (2018, 0), "r 건대추/냉동대추 5톤 이상 끝해": (2024, 0),
 "냉동대추 중국 5톤 미만 해 수": (3, 0), "냉동대추 중국 5톤 미만 해 단가 최소": (9, 0.5), "냉동대추 중국 5톤 미만 해 단가 최대": (13, 0.5), "r 건대추/냉동대추 5톤 미만 최대": (0.06, 0.005), "냉동대추 중국 톤 2018": (20.4, 0.05), "냉동대추 중국 톤 2020": (653.1, 0.05), "냉동대추 중국 톤 2021": (9.2, 0.05), "냉동대추 중국 톤 2022": (55.0, 0.05), "냉동대추 중국 톤 2024": (7.4, 0.05),
 "냉동대추 중국 단가 2020": (0.72, 0.005), "건대추 중국 톤 2017~2025 최소": (222, 0.5), "건대추 중국 톤 2017~2025 최대": (640, 0.5), "건대추 중국 단가 2017~2025 최소": (0.80, 0.005), "건대추 중국 단가 2017~2025 최대": (1.68, 0.005), "건대추 하한 원": (948, 0.5), "건대추 중국 몫 2017~2025": (0.96, 0.005),
}
def check(name, exp):
    bad, miss, ok = [], [], 0
    for k, (v, tol) in exp.items():
        if k not in CHK: miss.append(k); continue
        got = CHK[k]
        try: g = float(got)
        except (TypeError, ValueError): g = np.nan
        if not np.isfinite(g) or abs(g - v) > tol + 1e-9: bad.append((k, v, got))
        else: ok += 1
    print(f"[{name}] 대조 {len(exp)}건: 통과 {ok}, 불일치 {len(bad)}, 미등록 {len(miss)}")
    for b in bad: print("   불일치", b)
    for m in miss: print("   미등록", m)
    return bad, miss
RES = {"논문 A": check("논문 A", EXPECT_A)}
cited = set(EXPECT_A)
print("등록만 되고 인용되지 않은 라벨:", len([k for k in CHK if k not in cited]))
json.dump(CHK, open(os.path.join(O, "논문A_수치.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
json.dump({n: dict(bad=[list(map(str, b)) for b in r[0]], miss=r[1]) for n, r in RES.items()}, open(os.path.join(O, "논문A_검증_결과.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
n_bad = sum(len(r[0]) + len(r[1]) for r in RES.values())
assert n_bad == 0, f"논문 대조 실패 {n_bad}건 — outputs/논문A_검증_결과.json"
print("논문 A 대조 통과:", len(EXPECT_A), "건")

[논문 A] 대조 389건: 통과 389, 불일치 0, 미등록 0
등록만 되고 인용되지 않은 라벨: 229
논문 A 대조 통과: 389 건
